# EduHinglish: Gemma-3-4B-it QLoRA — v4, all subjects, resumable (T4)

Trains the Hinglish answer generator on **Biology, Chemistry, Physics, Maths and Social Science (Class 9 & 10)**.

**Resumable:** checkpoints go to Google Drive every `SAVE_STEPS` steps. When the Colab runtime disconnects,
reconnect and **Run all** again — training continues from the last checkpoint (same step, optimizer,
LR schedule and data order), it does not start over. Once training is finished, re-running skips
straight to the saved adapter.

> **Runtime**: Runtime → Change runtime type → **T4 GPU**

## 0. Install dependencies
Same pins as the old v3 run (OLD_04 notebook). **Restart the runtime once after the first install** (Runtime → Restart session), then Run all.

In [ ]:
!pip install -q -U \
    "bitsandbytes>=0.43.0" \
    "transformers>=4.47.0" \
    "peft>=0.14.0" \
    "accelerate>=1.0.0" \
    "datasets>=3.0.0" \
    "trl>=0.12.0,<0.20.0" \
    "huggingface_hub" \
    "kagglehub" \
    "evaluate" "rouge_score" "sacrebleu" "bert_score"

## 1. Settings — the only cell you normally edit

In [ ]:
BASE_DIR = "/content/drive/MyDrive/EduHinglish"
RUN_NAME = "gemma_eduhinglish_v4"

# Where the 5 dataset JSONs come from:
#   "drive"  -> {BASE_DIR}/data/<file>   (upload them from the repo's data/ folder once)
#   "kaggle" -> the Kaggle dataset published with scripts/publish_kaggle_dataset.py
DATA_SOURCE    = "drive"
KAGGLE_DATASET = "your-kaggle-username/eduhinglish-ncert-hinglish-qa"   # only for DATA_SOURCE="kaggle"
DATA_FILES = ["bio_9_10.json", "chem_9_10.json", "phys_9_10.json", "maths_9_10.json", "ss_9_10.json"]

# Start the NEW run from the v3 adapter (bio + ss) instead of from scratch. Only used when no
# v4 checkpoint exists yet; set to None to train from the plain base model.
WARM_START_ADAPTER = f"{BASE_DIR}/models/gemma_eduhinglish_v3/adapter"

NUM_EPOCHS = 2        # ~3,400 train rows / effective batch 8 ≈ 430 steps per epoch (~50 min/epoch on T4)
SAVE_STEPS = 50       # checkpoint every ~6 min, so a disconnect loses at most a few minutes
EVAL_STEPS = 50       # must equal SAVE_STEPS so the best checkpoint (lowest eval loss) can be kept

# Evaluation (section 8)
EVAL_BATCH = 8        # questions generated in parallel; lower to 4 if you hit CUDA OOM
GITHUB_REPO = "https://github.com/PersuasivePost/EduHinglish.git"   # for the project's word-level LID (CMI)

# Hugging Face Hub (needs a WRITE token in the HF_TOKEN secret).
# Separate repo from v3, so the model server keeps using v3 until v4 is proven better.
HF_REPO_ID = "persuasivepost/eduhinglish-gemma-3-4b-it-lora-v4"
PUSH_FINAL_ADAPTER_TO_HUB = True
# Also push every checkpoint to the Hub (folder "last-checkpoint"). Lets you resume even if Drive
# is unavailable, at the cost of a ~400 MB upload per save. Drive alone is usually enough.
PUSH_CHECKPOINTS_TO_HUB = False

CKPT_DIR    = f"{BASE_DIR}/models/{RUN_NAME}/checkpoints"
ADAPTER_DIR = f"{BASE_DIR}/models/{RUN_NAME}/adapter"

## 2. Authentication & Drive
Gemma is gated: accept the licence at https://huggingface.co/google/gemma-3-4b-it.
Add a **Write** token from https://huggingface.co/settings/tokens to Colab **Secrets** as `HF_TOKEN` (enable notebook access).
For `DATA_SOURCE="kaggle"` also add `KAGGLE_USERNAME` and `KAGGLE_KEY` secrets.

In [ ]:
import os
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
os.makedirs(CKPT_DIR, exist_ok=True)

os.environ["HF_FOR_COLAB"] = userdata.get("HF_FOR_COLAB")
login(token=os.environ["HF_FOR_COLAB"])

if DATA_SOURCE == "kaggle":
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("Logged in.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Logged in.


## 3. Training state — what has been done so far

In [ ]:
import json, glob
from transformers.trainer_utils import get_last_checkpoint

def find_last_checkpoint():
    last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
    if last is None and PUSH_CHECKPOINTS_TO_HUB:
        # Drive copy missing: try the checkpoint pushed to the Hub
        from huggingface_hub import snapshot_download
        try:
            path = snapshot_download(HF_REPO_ID, allow_patterns=["last-checkpoint/*"],
                                     local_dir=f"{CKPT_DIR}/_hub")
            if os.path.exists(f"{path}/last-checkpoint/trainer_state.json"):
                last = f"{path}/last-checkpoint"
        except Exception as e:
            print("No Hub checkpoint:", e)
    return last

LAST_CKPT = find_last_checkpoint()
TRAINING_DONE = os.path.exists(f"{ADAPTER_DIR}/adapter_config.json") and \
                os.path.exists(f"{BASE_DIR}/models/{RUN_NAME}/TRAINING_DONE")

if TRAINING_DONE:
    print(f"Training already finished. Final adapter: {ADAPTER_DIR}")
elif LAST_CKPT:
    st = json.load(open(f"{LAST_CKPT}/trainer_state.json"))
    print(f"Resuming from {LAST_CKPT}")
    print(f"  step {st['global_step']} / {st['max_steps']}  (epoch {st['epoch']:.2f})")
    losses = [h for h in st["log_history"] if "loss" in h]
    evals  = [h for h in st["log_history"] if "eval_loss" in h]
    if losses: print(f"  last train loss {losses[-1]['loss']:.4f}")
    if evals:  print(f"  last eval  loss {evals[-1]['eval_loss']:.4f}")
else:
    print("No checkpoint yet — starting a new run.")

No checkpoint yet — starting a new run.


## 4. Load the dataset (all five subjects)
Each record already has a `text` field rendered by `scripts/build_dataset.py` in the exact prompt
format `src/generator.py` uses at inference (context, question, optional intent line, instruction),
so the notebook does no prompt formatting of its own.

The train/eval split is decided by a hash of each record's `id`, so it is identical on every
session (needed for a correct resume) and stays stable when new chapters are added.

In [ ]:
import hashlib
from collections import Counter
from datasets import Dataset

if DATA_SOURCE == "kaggle":
    import kagglehub
    data_dir = kagglehub.dataset_download(KAGGLE_DATASET)
else:
    data_dir = f"{BASE_DIR}/data"

rows = []
for name in DATA_FILES:
    with open(f"{data_dir}/{name}", encoding="utf-8") as f:
        entries = json.load(f)
    print(f"{name:18s} {len(entries):5d} entries")
    rows.extend(entries)

def is_eval(rid, share=0.05):
    return int(hashlib.md5(rid.encode()).hexdigest(), 16) % 10_000 < share * 10_000

train_rows = [r for r in rows if not is_eval(r["id"])]
eval_rows  = [r for r in rows if is_eval(r["id"])]
print(f"\nTotal {len(rows)} | train {len(train_rows)} | eval {len(eval_rows)}")
print("By subject:", dict(Counter(r["subject"] for r in rows)))

train_dataset = Dataset.from_list([{"text": r["text"]} for r in train_rows]).shuffle(seed=42)
eval_dataset  = Dataset.from_list([{"text": r["text"]} for r in eval_rows])
print("\nSample:\n", train_dataset[0]["text"][:800])

bio_9_10.json        754 entries
chem_9_10.json       361 entries
phys_9_10.json       355 entries
maths_9_10.json      710 entries
ss_9_10.json        1424 entries

Total 3604 | train 3436 | eval 168
By subject: {'Biology': 754, 'Chemistry': 361, 'Physics': 355, 'Mathematics': 710, 'Social Science': 1424}

Sample:
 <start_of_turn>user
Context:
Feminist: awareness of women's rights and interests based on the belief of the social, economic and political equality of the genders. Ideology: system of ideas reflecting a particular social and political vision. Louise Otto-Peters founded a women's journal and subsequently a feminist political association.

Question:
'Feminist' aur 'ideology' shabdon ka kya matlab hai?

Intent Type: definition
Answer the question in Hinglish using only the given context.<end_of_turn>
<start_of_turn>model
Feminist ka matlab hai genders ki social, economic aur political barabari ke vishwaas pe based mahilaon ke haq aur interests ki jaagrukta. Ideology ek khaas s

## 5. Model + LoRA (QLoRA, fp16 — T4 has no bf16)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model

MODEL_ID = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
model.config.use_cache = False
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

if TRAINING_DONE:
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    print("Loaded finished adapter (no training needed).")
elif LAST_CKPT:
    # Fresh LoRA layers; trainer.train(resume_from_checkpoint=...) loads the checkpoint weights into them
    model = get_peft_model(model, lora_config)
    print("LoRA initialised — weights will be restored from", LAST_CKPT)
elif WARM_START_ADAPTER and os.path.exists(f"{WARM_START_ADAPTER}/adapter_config.json"):
    model = PeftModel.from_pretrained(model, WARM_START_ADAPTER, is_trainable=True)
    print("Warm start from", WARM_START_ADAPTER)
else:
    model = get_peft_model(model, lora_config)
    print("Training LoRA from scratch.")
model.print_trainable_parameters()

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Warm start from /content/drive/MyDrive/EduHinglish/models/gemma_eduhinglish_v3/adapter
trainable params: 32,788,480 || all params: 4,332,867,952 || trainable%: 0.7567


## 6. Train (or resume)
Safe to re-run at any time. If the runtime dies mid-training, reconnect and **Run all**.

- **Loss on the answer only:** the prompt (NCERT context + question) is masked out, so the model learns to *answer*, not to copy the textbook. Because of this, eval loss is not comparable with the v3 run's 1.41.
- **Best checkpoint kept:** eval runs every `EVAL_STEPS`; at the end the checkpoint with the lowest eval loss is loaded, not simply the last one.

In [ ]:
import math
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

assert SAVE_STEPS == EVAL_STEPS, "load_best_model_at_end needs SAVE_STEPS == EVAL_STEPS"

total_steps = math.ceil(len(train_dataset) / (2 * 4)) * NUM_EPOCHS
print(f"Planned optimizer steps: {total_steps}")

# Everything up to and including "<start_of_turn>model\n" gets label -100 (no loss).
RESPONSE_TEMPLATE = tokenizer.encode("<start_of_turn>model\n", add_special_tokens=False)
collator = DataCollatorForCompletionOnlyLM(response_template=RESPONSE_TEMPLATE, tokenizer=tokenizer)

# Sanity check: the trained tokens must be the answer, not the prompt
_b = collator([{"input_ids": tokenizer(train_dataset[0]["text"])["input_ids"]}])
_trained = [t for t, l in zip(_b["input_ids"][0].tolist(), _b["labels"][0].tolist()) if l != -100]
assert _trained, "Response template not found — every token is masked. Check the chat template."
print(f"Loss computed on {len(_trained)} of {len(_b['input_ids'][0])} tokens:")
print("  ", tokenizer.decode(_trained)[:200], "...")

sft_config = SFTConfig(
    output_dir=CKPT_DIR,
    dataset_text_field="text",
    max_length=1024,
    packing=False,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    warmup_steps=max(1, int(0.03 * total_steps)),
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False,

    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,          # newest checkpoint + best checkpoint stay on Drive
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    data_seed=42,
    report_to="none",

    push_to_hub=PUSH_CHECKPOINTS_TO_HUB,
    hub_model_id=HF_REPO_ID if PUSH_CHECKPOINTS_TO_HUB else None,
    hub_strategy="checkpoint",
    hub_private_repo=True,
)

if TRAINING_DONE:
    trainer = None
    print("Already trained — skipping.")
else:
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=collator,
        args=sft_config,
    )
    trainer.train(resume_from_checkpoint=LAST_CKPT)   # None = new run
    print("Best checkpoint:", trainer.state.best_model_checkpoint,
          "| best eval loss:", trainer.state.best_metric)

Planned optimizer steps: 860
Loss computed on 57 of 168 tokens:
   Feminist ka matlab hai genders ki social, economic aur political barabari ke vishwaas pe based mahilaon ke haq aur interests ki jaagrukta. Ideology ek khaas social aur political soch ko dikhane wala v ...


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing_kwargs` is deprecated and will be removed in v6. Use `activation_checkpointing_kwargs` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Adding EOS to train dataset:   0%|          | 0/3436 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3436 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3436 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/168 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/168 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/168 [00:00<?, ? examples/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
50,0.917052,1.002221
100,0.954799,1.008444
150,0.907583,0.983879
200,1.073749,0.974310
250,1.005564,0.958409
300,0.882355,0.956905
350,0.961453,0.930029
400,0.988170,0.908546
450,0.509124,0.947002
500,0.463354,0.948081


Best checkpoint: /content/drive/MyDrive/EduHinglish/models/gemma_eduhinglish_v4/checkpoints/checkpoint-850 | best eval loss: 0.9034384489059448


## 7. Save the best adapter to Drive

In [ ]:
if not TRAINING_DONE:
    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    metrics = trainer.evaluate()
    metrics["eval_perplexity"] = math.exp(metrics["eval_loss"])
    metrics["best_checkpoint"] = trainer.state.best_model_checkpoint
    with open(f"{ADAPTER_DIR}/eval_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    open(f"{BASE_DIR}/models/{RUN_NAME}/TRAINING_DONE", "w").write(json.dumps(metrics))
    TRAINING_DONE = True
    print("Saved", ADAPTER_DIR)
    print(json.dumps(metrics, indent=2))
else:
    print("Adapter already at", ADAPTER_DIR)

Training Loss,Validation Loss,Step
0.539340,0.903438,860


Saved /content/drive/MyDrive/EduHinglish/models/gemma_eduhinglish_v4/adapter
{
  "eval_loss": 0.9034384489059448,
  "eval_perplexity": 2.4680748873215412,
  "best_checkpoint": "/content/drive/MyDrive/EduHinglish/models/gemma_eduhinglish_v4/checkpoints/checkpoint-850"
}


## 8. Evaluate on the held-out set
Generates an answer for every held-out question (same prompt builder as `src/generator.py`) and scores it against the reference answer:

| Metric | What it measures |
|---|---|
| **ROUGE-L** | longest common word sequence with the reference (content overlap) |
| **chrF++** | character + word n-gram F-score; tolerant of Hinglish spelling variants (*hai / he*) |
| **BERTScore F1** | semantic similarity (multilingual BERT), credits paraphrases |
| **Key accuracy** | MCQ / assertion-reason option and true/false verdict match the reference |
| **CMI** | Code-Mixing Index of generated vs reference answers, computed with the project's `WordLevelLID` — same numbers the pipeline reports. A small **CMI gap** means the model mixes Hindi and English like the curated answers |

Scores are reported overall, by subject and by intent, and saved to `eval_report.json` (uploaded with the adapter).
Predictions are cached on Drive, so a disconnect mid-evaluation continues where it stopped. Takes ~15–20 min on a T4.

In [ ]:
import re, sys, subprocess
import evaluate
import pandas as pd

# ── Project LID for CMI (cloned from GitHub; script_detector.py has no heavy deps) ──
REPO = "/content/EduHinglish"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", GITHUB_REPO, REPO], check=True)
sys.path.insert(0, f"{REPO}/src")
from script_detector import WordLevelLID
LID = WordLevelLID()

def cmi(text):
    tags = [LID.identify_word(w) for w in text.split()]
    hi, en = tags.count("HI"), tags.count("EN")
    return round((1 - max(hi, en) / (hi + en)) * 100, 1) if hi + en else 0.0

# ── Generation (same prompt as src/generator.py) ──
model = trainer.model if trainer is not None else model
model.eval()
model.config.use_cache = True
END_TURN = tokenizer.convert_tokens_to_ids("<end_of_turn>")

def build_prompt(context, question, intent=None, language="hinglish"):
    intent_line = f"Intent Type: {intent}\n" if intent else ""
    instruction = f"Answer the question in {'Hinglish' if language == 'hinglish' else 'English'} using only the given context."
    return f"Context:\n{context}\n\nQuestion:\n{question}\n\n{intent_line}{instruction}"

@torch.no_grad()
def generate(prompts, max_new_tokens=256):
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                           add_generation_prompt=True) for p in prompts]
    tokenizer.padding_side = "left"
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    tokenizer.padding_side = "right"
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=[tokenizer.eos_token_id, END_TURN])
    return [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

def ask(context, question, intent=None):
    return generate([build_prompt(context, question, intent)])[0]

# ── Generate all held-out answers (cached on Drive, resumable) ──
PRED_FILE = f"{BASE_DIR}/models/{RUN_NAME}/eval_predictions.jsonl"   # delete this file after retraining
preds = {}
if os.path.exists(PRED_FILE):
    with open(PRED_FILE) as f:
        for line in f:
            d = json.loads(line)
            preds[d["id"]] = d["prediction"]
todo = [r for r in eval_rows if r["id"] not in preds]
print(f"{len(preds)} cached, {len(todo)} to generate")
for i in range(0, len(todo), EVAL_BATCH):
    batch = todo[i:i + EVAL_BATCH]
    outs = generate([build_prompt(r["ncert_context"], r["question_hinglish"], r["intent"]) for r in batch])
    with open(PRED_FILE, "a") as f:
        for r, p in zip(batch, outs):
            preds[r["id"]] = p
            f.write(json.dumps({"id": r["id"], "prediction": p}, ensure_ascii=False) + "\n")
    print(f"  {min(i + EVAL_BATCH, len(todo))}/{len(todo)}")

# ── Metrics ──
rouge, chrf, bertscore = evaluate.load("rouge"), evaluate.load("chrf"), evaluate.load("bertscore")
bs = bertscore.compute(predictions=[preds[r["id"]] for r in eval_rows],
                       references=[r["answer_hinglish"] for r in eval_rows],
                       model_type="bert-base-multilingual-cased", batch_size=16)
BERT_F1 = {r["id"]: f for r, f in zip(eval_rows, bs["f1"])}

KEY_INTENTS = {"mcq", "assertion_reason", "true_false"}
def answer_key(intent, text):
    t = text.strip().lower()
    if intent in ("mcq", "assertion_reason"):
        m = re.search(r"\(\s*(iv|v|i{1,3}|[a-d])\s*\)", t)
        return m.group(1) if m else None
    if intent == "true_false":
        m = re.search(r"\b(true|false|sahi|galat)\b", t)
        return {"sahi": "true", "galat": "false"}.get(m.group(1), m.group(1)) if m else None
    return None

def score(rs):
    P = [preds[r["id"]] for r in rs]
    R = [r["answer_hinglish"] for r in rs]
    cp, cr = [cmi(p) for p in P], [cmi(x) for x in R]
    out = {
        "n": len(rs),
        "rougeL": round(rouge.compute(predictions=P, references=R)["rougeL"] * 100, 2),
        "chrf++": round(chrf.compute(predictions=P, references=[[x] for x in R], word_order=2)["score"], 2),
        "bertscore_f1": round(sum(BERT_F1[r["id"]] for r in rs) / len(rs) * 100, 2),
        "cmi_pred": round(sum(cp) / len(cp), 1),
        "cmi_ref": round(sum(cr) / len(cr), 1),
        "cmi_gap": round(sum(abs(a - b) for a, b in zip(cp, cr)) / len(cp), 1),
    }
    keyed = [(answer_key(r["intent"], preds[r["id"]]), answer_key(r["intent"], r["answer_hinglish"]))
             for r in rs if r["intent"] in KEY_INTENTS]
    keyed = [(p, g) for p, g in keyed if g is not None]
    if keyed:
        out["key_accuracy"] = round(sum(p == g for p, g in keyed) / len(keyed) * 100, 1)
        out["n_key"] = len(keyed)
    return out

def group(field):
    g = {}
    for r in eval_rows:
        g.setdefault(r[field], []).append(r)
    return {k: score(v) for k, v in sorted(g.items())}

train_metrics = json.loads(open(f"{BASE_DIR}/models/{RUN_NAME}/TRAINING_DONE").read())
report = {
    "run": RUN_NAME,
    "eval_loss": train_metrics.get("eval_loss"),
    "eval_perplexity": train_metrics.get("eval_perplexity"),
    "overall": score(eval_rows),
    "by_subject": group("subject"),
    "by_intent": group("intent"),
}
with open(f"{ADAPTER_DIR}/eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"\neval_loss {report['eval_loss']:.4f} | perplexity {report['eval_perplexity']:.2f}")
print("\n── Overall ──");     display(pd.DataFrame([report["overall"]]))
print("── By subject ──");   display(pd.DataFrame(report["by_subject"]).T)
print("── By intent ──");    display(pd.DataFrame(report["by_intent"]).T)

# One example per subject for a quick read
seen = set()
for r in eval_rows:
    if r["subject"] in seen:
        continue
    seen.add(r["subject"])
    print(f"\n=== {r['subject']} | {r['chapter']} [{r['intent']}]")
    print("Q:  ", r["question_hinglish"])
    print("GEN:", preds[r["id"]])
    print("REF:", r["answer_hinglish"])

[OK] WordLevelLID initialized
0 cached, 168 to generate
  8/168
  16/168
  24/168
  32/168
  40/168
  48/168
  56/168
  64/168
  72/168
  80/168
  88/168
  96/168
  104/168
  112/168
  120/168
  128/168
  136/168
  144/168
  152/168
  160/168
  168/168


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

... (6 outputs left out)

,n,rougeL,chrf++,bertscore_f1,cmi_pred,cmi_ref,cmi_gap,key_accuracy,n_key
Biology,36.0,53.77,57.14,82.37,43.2,43.3,3.9,50.0,2.0
Chemistry,17.0,48.90,51.39,81.27,38.6,38.5,4.3,0.0,2.0
Mathematics,28.0,44.67,44.72,78.99,29.0,28.1,8.7,0.0,1.0
Physics,12.0,34.99,42.04,76.12,42.1,39.6,4.9,0.0,1.0
Social Science,75.0,50.03,54.18,80.67,40.8,40.6,4.3,100.0,6.0


── By intent ──


,n,rougeL,chrf++,bertscore_f1,cmi_pred,cmi_ref,cmi_gap,key_accuracy,n_key
analytical,20.0,37.12,46.66,77.14,43.0,43.6,4.1,NaN,NaN
application_reallife,5.0,54.84,52.03,79.71,42.8,44.0,4.7,NaN,NaN
compare_concepts,10.0,45.33,51.03,78.71,40.2,41.5,5.1,NaN,NaN
definition,14.0,56.30,58.75,82.43,44.9,42.9,4.5,NaN,NaN
describe_function,3.0,67.11,70.48,85.29,45.0,43.3,2.6,NaN,NaN
explain_concept,50.0,47.41,54.37,80.24,41.9,42.1,3.8,NaN,NaN
fill_blank,1.0,64.00,53.14,81.89,11.1,16.7,5.6,NaN,NaN
give_example,2.0,26.10,38.64,73.26,40.8,37.6,3.7,NaN,NaN
list_enumerate,7.0,50.29,54.21,80.90,35.6,35.9,6.0,NaN,NaN
mcq,6.0,50.46,49.48,81.70,38.4,38.4,2.6,33.3,6.0



=== Biology | Cell: The Building Block of Life [compare_concepts]
Q:   Diffusion aur osmosis mein kya difference hai?
GEN: Diffusion mein particles apne concentration gradient ke hisaab se faile hote hain; isme sirf particles move karte hain, jaise paani mein rang ka boond ya hawa mein fragrance. Osmosis ek special case hai jisme sirf paani moves, selectively permeable membrane ke through, aur iska koi concentration gradient nahi hota.
REF: Diffusion mein kisi bhi substance ke particles higher concentration se lower concentration ki taraf move karte hain, aur iske liye membrane zaroori nahi hai. Osmosis specifically paani ka diffusion hai jo selectively permeable membrane ke through hota hai.

=== Chemistry | Exploring Mixtures and their Separation [definition]
Q:   Solute aur solvent kya hote hain? Sugar solution mein inhe pehchano.
GEN: Solution mein solute woh padaarth hai jo ghulta hai, aur solvent woh hai jo solute ko ghulte hai. Sugar solution mein sugar solute hai aur paani sol

## 9. Push the adapter + evaluation report to the Hugging Face Hub
Uploads the LoRA adapter (~130 MB), `eval_metrics.json` and `eval_report.json`, with the scores in the model card.
Goes to a **separate v4 repo**: switch `HF_ADAPTER_REPO` in notebook 07 to it only once v4 beats v3.

In [ ]:
from huggingface_hub import HfApi

# Bypassed the safety check to allow pushing to 'persuasivepost'
# assert "persuasivepost" not in HF_REPO_ID, "Set HF_REPO_ID in the Settings cell first"
o = report["overall"]
subject_rows = "\n".join(
    f"| {s} | {m['n']} | {m['rougeL']} | {m['chrf++']} | {m['bertscore_f1']} | {m['cmi_pred']} / {m['cmi_ref']} |"
    for s, m in report["by_subject"].items())

CARD = f'''---
base_model: google/gemma-3-4b-it
library_name: peft
license: gemma
language: [en, hi]
tags: [lora, qlora, hinglish, education, ncert, code-mixing]
pipeline_tag: text-generation
---
# EduHinglish — Gemma-3-4B-it LoRA ({RUN_NAME})

LoRA adapter that answers NCERT Class 9–10 questions (Biology, Chemistry, Physics, Mathematics,
Social Science) in Roman-script Hinglish, grounded in a retrieved textbook passage.

Trained with QLoRA (4-bit NF4, r=16, alpha=32), loss on answer tokens only, on {len(train_rows)} curated
QA pairs; best checkpoint by eval loss. Evaluated on {len(eval_rows)} held-out pairs.

## Evaluation (held-out set)
| Metric | Score |
|---|---|
| Eval loss / perplexity | {report['eval_loss']:.4f} / {report['eval_perplexity']:.2f} |
| ROUGE-L | {o['rougeL']} |
| chrF++ | {o['chrf++']} |
| BERTScore F1 (mBERT) | {o['bertscore_f1']} |
| Key accuracy (MCQ / A-R / T-F) | {o.get('key_accuracy', 'n/a')} |
| CMI generated / reference (gap) | {o['cmi_pred']} / {o['cmi_ref']} ({o['cmi_gap']}) |

| Subject | n | ROUGE-L | chrF++ | BERTScore | CMI gen / ref |
|---|---|---|---|---|---|
{subject_rows}

Full breakdown by subject and intent: `eval_report.json`.

## Prompt format (single user turn)
```
Context:
<NCERT passage>

Question:
<student question>

Intent Type: <intent>          # optional line
Answer the question in Hinglish using only the given context.
```

## Usage
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
tok = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
base = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it", device_map="auto")
model = PeftModel.from_pretrained(base, "{HF_REPO_ID}")
```
Use of this adapter is subject to the Gemma Terms of Use.
'''

if PUSH_FINAL_ADAPTER_TO_HUB:
    with open(f"{ADAPTER_DIR}/README.md", "w") as f:
        f.write(CARD)
    api = HfApi()
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO_ID, folder_path=ADAPTER_DIR,
                      commit_message=f"{RUN_NAME} adapter + eval report")
    print(f"Pushed: https://huggingface.co/{HF_REPO_ID}")

Pushed: https://huggingface.co/persuasivepost/eduhinglish-gemma-3-4b-it-lora-v4


## 10. Test the new model — questions from all 5 subjects
New questions (not from the dataset), two per subject, each with a short NCERT-style passage, since this notebook has no retriever.
Add your own with `ask(context, question, intent)`.

In [ ]:
TEST_QUESTIONS = [
    # ── Biology ──
    ("Biology",
     "Photosynthesis is the process by which green plants prepare their own food (carbohydrates) from carbon dioxide and water "
     "in the presence of sunlight and chlorophyll. Oxygen is released as a by-product. "
     "6CO₂ + 12H₂O → C₆H₁₂O₆ + 6O₂ + 6H₂O (in the presence of sunlight and chlorophyll).",
     "Photosynthesis kaise hota hai aur isme kaunsi gas nikalti hai?", "explain_concept"),
    ("Biology",
     "The cell is the basic structural and functional unit of all living organisms. Robert Hooke first observed cells in 1665 "
     "in a thin slice of cork using a self-designed microscope.",
     "Cell kise kehte hain aur ise sabse pehle kisne dekha tha?", "definition"),

    # ── Chemistry ──
    ("Chemistry",
     "Reactions in which heat is released along with the formation of products are called exothermic reactions, for example "
     "burning of natural gas: CH₄ + 2O₂ → CO₂ + 2H₂O + heat, and respiration. Reactions in which energy is absorbed are called "
     "endothermic reactions, for example the decomposition of calcium carbonate on heating: CaCO₃ → CaO + CO₂.",
     "Exothermic aur endothermic reaction mein kya fark hai? Ek-ek example do.", "compare_concepts"),
    ("Chemistry",
     "The pH scale measures the hydrogen ion concentration of a solution from 0 to 14. A pH of 7 is neutral. "
     "A pH less than 7 indicates an acidic solution and a pH more than 7 indicates a basic (alkaline) solution.",
     "Sahi ya galat: Jis solution ka pH 9 hai woh acidic hota hai.", "true_false"),

    # ── Physics ──
    ("Physics",
     "Ohm's law states that the electric current flowing through a metallic wire is directly proportional to the potential "
     "difference V across its ends, provided its temperature remains the same. That is, V = IR, where R is the resistance of "
     "the conductor. The SI unit of resistance is ohm (Ω).",
     "Ohm ka law kya hai?", "explain_concept"),
    ("Physics",
     "The average speed of an object is obtained by dividing the total distance travelled by the total time taken. "
     "Average speed = total distance travelled / total time taken. The SI unit of speed is metre per second (m/s); "
     "km/h is also commonly used.",
     "Ek car 3 ghante mein 150 km chalti hai. Uski average speed kitni hogi?", "numerical_problem"),

    # ── Maths ──
    ("Maths",
     "Pythagoras theorem: In a right triangle, the square of the hypotenuse is equal to the sum of the squares of the other "
     "two sides. If the sides are a and b and the hypotenuse is c, then c² = a² + b².",
     "Ek right triangle ki do sides 6 cm aur 8 cm hain. Hypotenuse kitna hoga?", "numerical_problem"),
    ("Maths",
     "The roots of the quadratic equation ax² + bx + c = 0 are given by x = (−b ± √(b² − 4ac)) / 2a. The value b² − 4ac is "
     "called the discriminant. If b² − 4ac > 0 the equation has two distinct real roots, if b² − 4ac = 0 it has two equal "
     "real roots, and if b² − 4ac < 0 it has no real roots.",
     "Discriminant se kaise pata chalta hai ki quadratic equation ke roots real hain ya nahi?", "explain_concept"),

    # ── Social Science ──
    ("Social Science",
     "Democracy is a form of government in which the rulers are elected by the people. In a democracy, the final decision-"
     "making power rests with those elected by the people, elections are free and fair, each adult citizen has one vote and "
     "each vote has one value, and the government rules within limits set by constitutional law and citizens' rights.",
     "Democracy kya hoti hai? Iske main features batao.", "definition"),
    ("Social Science",
     "The French Revolution began in 1789 with the storming of the Bastille on 14 July. It ended the rule of the monarchy in "
     "France and spread the ideas of liberty, equality and fraternity, which inspired many later movements across Europe.",
     "French Revolution ke teen main ideals kya the aur yeh kab shuru hui?", "list_enumerate"),
]

print("── EduHinglish v4 — test questions ──")
for subject, context, question, intent in TEST_QUESTIONS:
    answer = ask(context, question, intent)
    print(f"\n[{subject} | {intent}]")
    print("Q:  ", question)
    print("A:  ", answer)
    print(f"CMI: {cmi(answer)}%")

── EduHinglish v4 — test questions ──

[Biology | explain_concept]
Q:   Photosynthesis kaise hota hai aur isme kaunsi gas nikalti hai?
A:   Green plants sunlight aur chlorophyll ki presence mein carbon dioxide aur paani se carbohydrates banate hain, aur oxygen ko by-product ke roop mein chhodte hain. Equation: 6CO₂ + 12H₂O → C₆H₁₂O₆ + 6O₂ + 6H₂O.
CMI: 38.7%

[Biology | definition]
Q:   Cell kise kehte hain aur ise sabse pehle kisne dekha tha?
A:   Cell ek living organism ki basic structural aur functional unit hai. Robert Hooke ne 1665 mein apne self-designed microscope se cork ki ek patli slice mein cells dekhe, aur yeh sabse pehla insaan tha jinka yeh observation tha.
CMI: 48.6%

[Chemistry | compare_concepts]
Q:   Exothermic aur endothermic reaction mein kya fark hai? Ek-ek example do.
A:   Exothermic reactions mein products ke saath garmi nikalti hai, jaise natural gas ka jalna (CH₄ + 2O₂ → CO₂ + 2H₂O + heat) aur saans. Endothermic reactions mein urja li hoti hai, jaise garam kachc